# Experiment 01 - Python and Pandas Review

**Purpose:** Verify Python exponentiation, GPU memory capacity units, and Pandas table/CSV behavior used by this project's environment checks and experiment reports.

This is a **small, reproducible investigation of APIs and data representation**, not an inference benchmark. It does not measure model performance, CUDA utilization, or inference costs. The unit-conversion exercises use a fixed reference value so they work on CPU-only machines. A separate, optional section inspects the GPU on the computer running the notebook. The Pandas example uses **synthetic records**, not experimental measurements.

**Execution:** Run the cells in order from a fresh Python kernel. Python and Pandas are required; PyTorch is optional for the live GPU inspection. No internet access, GPU, input files, or changes to the repository are required.

## 1. Reproducibility and environment

Record the runtime used to perform this review. Versions matter when reproducing an experiment, but the output below describes **the current machine**, not necessarily the environment documented in the project's `README.md`.

In [1]:
import platform
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

try:
    import torch
except ImportError:
    torch = None

print(f"Python: {platform.python_version()}")
print(f"Pandas: {pd.__version__}")
print(f"PyTorch: {torch.__version__ if torch is not None else 'not installed'}")
print(f"CUDA available: {bool(torch is not None and torch.cuda.is_available())}")

Python: 3.12.3
Pandas: 3.0.6
PyTorch: 2.14.0+cu132
CUDA available: True


## 2. Inspecting GPU memory capacity

`torch.cuda.is_available()` reports whether CUDA is usable by the installed PyTorch runtime. `torch.cuda.device_count()` returns the number of visible CUDA devices. `torch.cuda.get_device_properties(device)` returns information about a specific CUDA device, and its `total_memory` attribute is an **integer number of bytes**.

Device index `0` refers to the first CUDA device **visible to the current process**. It need not identify the first physical GPU in the computer if device visibility is configured differently. The inspection is skipped when CUDA is unavailable; this is not an experiment failure.

In [2]:
gpu_capacity_bytes = None


def describe_first_cuda_device():
    if torch is None:
        print("Skipped: PyTorch is not installed.")
        return None
    if not torch.cuda.is_available():
        print("Skipped: this PyTorch runtime cannot access CUDA.")
        return None
    if torch.cuda.device_count() == 0:
        print("Skipped: no CUDA device is visible.")
        return None

    properties = torch.cuda.get_device_properties(0)
    capacity_bytes = properties.total_memory
    assert isinstance(capacity_bytes, int) and capacity_bytes > 0

    print(f"CUDA device: {properties.name}")
    print(f"Total device memory: {capacity_bytes:,} bytes")
    print(f"Python type: {type(capacity_bytes).__name__}")
    return capacity_bytes


gpu_capacity_bytes = describe_first_cuda_device()

CUDA device: NVIDIA RTX A3000 12GB Laptop GPU
Total device memory: 12,491,292,672 bytes
Python type: int


`total_memory` describes reported device capacity. It is **not** currently free memory, memory consumed by a model, peak GPU usage, or a guarantee that an allocation of that size will succeed. The inference benchmark uses PyTorch allocator statistics for a different measurement purpose. Don't treat capacity and allocated(reserved) memory as interchangeable.

In the original version of this notebook, the recorded capacity was `12,491,292,672` bytes. We retain that number solely as a **fixed historical example**; it is not presented as a fresh hardware reading.

## 3. Exponentiation and storage units

Python uses the `**` operator for exponentiation. For example, `2 ** 3` means 2 multiplied by itself three times. This explains why `1024 ** 3` represents the number of bytes in one **gibibyte (GiB)**.

In [3]:
exponentiation_examples = (
    ("2 ** 3", 2**3),
    ("1024 ** 2", 1024**2),
    ("1024 ** 3", 1024**3),
)

for expression, result in exponentiation_examples:
    print(f"{expression} = {result:,}")

assert 2**3 == 8
assert 1024**2 == 1_048_576
assert 1024**3 == 1_073_741_824


2 ** 3 = 8
1024 ** 2 = 1,048,576
1024 ** 3 = 1,073,741,824


### 3.1 Decimal versus binary prefixes

Decimal units use powers of **1000**; binary units use powers of **1024**. The symbols aren't interchangeable.

| Unit | Definition in bytes | Unit | Definition in bytes |
|:--|--:|:--|--:|
| kB | 1000 | KiB | 1024 |
| MB | 1000² | MiB | 1024² |
| GB | 1000³ | GiB | 1024³ |

To convert a count of bytes, divide by the number of bytes in the target unit. Dividing by `1024 ** 3` produces **GiB**, not GB. Both conversions describe the same capacity using different scales; neither changes the underlying hardware.

The diagram from the original investigation remains available in the repository at [`notebooks/images/data-storage-units-hierarchy.png`](images/data-storage-units-hierarchy.png). The table and code below are sufficient even when the notebook is downloaded by itself.

In [4]:
REFERENCE_MEMORY_BYTES = 12_491_292_672  # Fixed historical example


def convert_bytes(byte_size):
    """Return decimal and binary storage-unit conversions of a byte count."""
    if not isinstance(byte_size, int) or byte_size < 0:
        raise ValueError("byte_count must be a non-negative integer")

    return {
        "bytes": byte_size,
        "kB": byte_size / (1000**1),
        "MB": byte_size / (1000**2),
        "GB": byte_size / (1000**3),
        "KiB": byte_size / (1024**1),
        "MiB": byte_size / (1024**2),
        "GiB": byte_size / (1024**3),
    }


memory_size_units = convert_bytes(REFERENCE_MEMORY_BYTES)
for unit in ("bytes", "kB", "MB", "GB", "KiB", "MiB", "GiB"):
    print(f"{unit:>5}: {memory_size_units[unit]:,.6f}")

assert memory_size_units["bytes"] == REFERENCE_MEMORY_BYTES
assert abs(memory_size_units["GB"] - 12.491292672) < 1e-10
assert abs(memory_size_units["GiB"] - 11.6334228515625) < 1e-10
assert memory_size_units["GB"] > memory_size_units["GiB"]

bytes: 12,491,292,672.000000
   kB: 12,491,292.672000
   MB: 12,491.292672
   GB: 12.491293
  KiB: 12,198,528.000000
  MiB: 11,912.625000
  GiB: 11.633423


### 3.2 Applying the conversion to actual hardware, when available

The following cell uses the live device reading only when one was successfully obtained. It does not overwrite the historical example or claim that the results are a GPU memory benchmark.

In [5]:
if gpu_capacity_bytes is None:
    print("Live GPU conversion skipped; the fixed reference example was verified above.")
else:
    gpu_units = convert_bytes(gpu_capacity_bytes)
    print(f"GPU capacity: {gpu_units['GB']:.3f} GB")
    print(f"GPU capacity: {gpu_units['GiB']:.3f} GiB")
    print("Both values refer to the same device capacity.")

GPU capacity: 12.491 GB
GPU capacity: 11.633 GiB
Both values refer to the same device capacity.


## 4. Pandas DataFrame and the row index

A `DataFrame` has **columns** and an **index**. The default `RangeIndex` labels rows as `0`, `1`, and so forth. Displaying that index does not make it an ordinary data column.

The next table deliberately contains **invented demonstration values**. The times below are not readings from `results/cpu_gpu_baseline.csv`, are not a comparison of real model configurations, and must not be used as performance evidence.

In [6]:
example_df = pd.DataFrame(
    [
        {"model": "example-model", "device": "CPU", "latency_seconds": 2.4},
        {"model": "example-model", "device": "CUDA", "latency_seconds": 0.8},
    ]
)

print(f"Columns: {list(example_df.columns)}")
print(f"Index: {example_df.index.tolist()}")

assert "index" not in example_df.columns
assert example_df.index.tolist() == [0, 1]

example_df

Columns: ['model', 'device', 'latency_seconds']
Index: [0, 1]


,model,device,latency_seconds
0,example-model,CPU,2.4
1,example-model,CUDA,0.8


## 5. `DataFrame.to_string()`

`DataFrame.to_string(index=...)` returns a **formatted Python string**, suitable for terminal output and inspection. Setting `index=False` hides row labels from that string; it does **not** delete or reset the DataFrame index. The default for this method includes the index.

In [7]:
text_with_index = example_df.to_string(index=True)
text_without_index = example_df.to_string(index=False)

print("With index:\n" + text_with_index)
print("\nWithout index:\n" + text_without_index)

assert isinstance(text_with_index, str)
assert isinstance(text_without_index, str)
assert example_df.index.tolist() == [0, 1]  # The DataFrame was not modified.
assert list(example_df.columns) == ["model", "device", "latency_seconds"]

With index:
           model device  latency_seconds
0  example-model    CPU              2.4
1  example-model   CUDA              0.8

Without index:
        model device  latency_seconds
example-model    CPU              2.4
example-model   CUDA              0.8


## 6. `DataFrame.to_csv()` and `index=False`

`DataFrame.to_csv(path_or_buf=None, index=...)` serializes rows and column headings as **CSV**. When no destination is supplied, it returns the CSV as a string. When a file path or writable buffer is supplied, it writes to that destination and returns `None`.

By default, `to_csv()` also writes the DataFrame index. With the unnamed default index this produces an initial blank header field, which can be interpreted as an extra column on import. `index=False` omits these row labels, which is usually appropriate for measurement records whose rows are already described by explicit columns.

In [8]:
csv_default = example_df.to_csv()
csv_with_index = example_df.to_csv(index=True)
csv_without_index = example_df.to_csv(index=False)

print("CSV with index (default):\n" + csv_with_index)
print("CSV without index:\n" + csv_without_index)

assert isinstance(csv_without_index, str)
assert csv_default == csv_with_index
assert csv_with_index.startswith(",model,device,latency_seconds\n")
assert csv_without_index.startswith("model,device,latency_seconds\n")
assert csv_without_index.count("\n") == len(example_df) + 1

CSV with index (default):
,model,device,latency_seconds
0,example-model,CPU,2.4
1,example-model,CUDA,0.8

CSV without index:
model,device,latency_seconds
example-model,CPU,2.4
example-model,CUDA,0.8



### 6.1 Reading the CSV back

`pd.read_csv()` reads CSV into a new DataFrame. `StringIO` lets us test this in memory without modifying files in the repository. This is a **round-trip verification**, not just a visual comparison of printouts.

When an unnamed index has been written, Pandas normally loads it as an additional column named `Unnamed: 0`. When the index was intentionally omitted, Pandas creates a fresh index for the imported data.

In [9]:
loaded_without_index = pd.read_csv(StringIO(csv_without_index))
loaded_with_index = pd.read_csv(StringIO(csv_with_index))

print(f"Columns with index exported: {list(loaded_with_index.columns)}")
print(f"Columns without index exported: {list(loaded_without_index.columns)}")

assert "Unnamed: 0" in loaded_with_index.columns
assert "Unnamed: 0" not in loaded_without_index.columns

pd.testing.assert_frame_equal(loaded_without_index, example_df)

# If the source index really was needed, it can be restored on import.
restored_index_df = pd.read_csv(StringIO(csv_with_index), index_col=0)
pd.testing.assert_frame_equal(restored_index_df, example_df)

print("CSV round-trip checks passed.")

Columns with index exported: ['Unnamed: 0', 'model', 'device', 'latency_seconds']
Columns without index exported: ['model', 'device', 'latency_seconds']
CSV round-trip checks passed.


### 6.2 When preserving an index is a good choice

`index=False` is not a universal rule. A meaningful index such as a device identifier may need to be exported. If it is omitted, that information can be lost unless it is also stored as a regular column. The decision depends on the data schema.

In [10]:
device_indexed_df = example_df.set_index("device")

saved_named_index = device_indexed_df.to_csv(index=True)
reloaded_named_index = pd.read_csv(StringIO(saved_named_index), index_col="device")

pd.testing.assert_frame_equal(reloaded_named_index, device_indexed_df)

print("CSV with a meaningful, named index:\n" + saved_named_index)
print("Named-index round-trip check passed.")

CSV with a meaningful, named index:
device,model,latency_seconds
CPU,example-model,2.4
CUDA,example-model,0.8

Named-index round-trip check passed.


### 6.3 Writing an actual CSV file without changing repository results

`to_csv(path, index=False)` writes a file and returns `None`. To verify the behavior safely, this cell uses a temporary directory that is deleted automatically. It does **not** overwrite real measurement results under `results/`.

In [11]:
with TemporaryDirectory() as temp_dir:
    output_path = Path(temp_dir) / "example_records.csv"
    write_return_value = example_df.to_csv(output_path, index=False)

    assert write_return_value is None
    assert output_path.is_file()
    
    pd.testing.assert_frame_equal(pd.read_csv(output_path), example_df)
    
    print(f"Temporary CSV verified: {output_path.name}")

print("Temporary file was cleaned up.")

Temporary CSV verified: example_records.csv
Temporary file was cleaned up.


## 7. Findings and project relevance

• **Memory capacity:** `torch.cuda.get_device_properties(0).total_memory` provides total device capacity in bytes, if a CUDA device is available. The live reading is conditional, while the fixed historical value allows the unit conversions to be independently verified.

• **Exponentiation:** `1024 ** 3` equals `1,073,741,824`, the number of bytes per GiB. Therefore `total_memory / (1024 ** 3)` must be labeled **GiB**, not GB. `1000 ** 3` is the corresponding divisor for GB.

• **Pandas display:** `DataFrame.to_string(index=False)` hides row index labels in the rendered string but leaves the original DataFrame unchanged.

• **Pandas export:** `DataFrame.to_csv(index=False)` omits index labels from the CSV. A round-trip using `pd.read_csv` confirms that it prevents an unnecessary `Unnamed: 0` column for an unnamed default index. A meaningful index should instead be explicitly preserved or converted to a column.

• **Boundary of this experiment:** It checks representation and serialization behavior only. It does not establish tokenizer efficiency, model output quality, CPU/GPU speedups, inference memory consumption, or token cost. Those conclusions require measurements from the dedicated project experiments.

## 8. API reference

| Call or operator | Input used here | Result | Official reference |
|:--|:--|:--|:--|
| `torch.cuda.is_available()` | No arguments | `bool` | [PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.cuda.is_available.html) |
| `torch.cuda.device_count()` | No arguments | `int` | [PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.cuda.device_count.html) |
| `torch.cuda.get_device_properties(device)` | Visible CUDA device index | Device properties object | [PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.cuda.get_device_properties.html) |
| `.total_memory` | Attribute of device properties | Capacity in bytes (`int`) | [PyTorch CUDA device properties](https://docs.pytorch.org/docs/stable/cuda.html) |
| `**` | Base and exponent | Exponentiation result | [Python reference](https://docs.python.org/3/reference/expressions.html#the-power-operator) |
| `DataFrame.to_string(index=...)` | `index=True` or `False` | `str` | [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_string.html) |
| `DataFrame.to_csv(path_or_buf=None, index=...)` | Optional destination and index flag | `str` if no destination; otherwise `None` | [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) |
| `pd.read_csv(...)` | File path or readable buffer | `DataFrame` | [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) |

The notebook uses assertions and CSV round trips to verify behavior. If any assertion fails in a different library version or runtime, investigate the discrepancy before treating the output as a valid result.